# xains Counterfactual Narratives: End-to-End on German Credit

A *counterfactual narrative* describes one recipe to flip a model's prediction -- which features would need to change, and to what. This notebook shows the realistic workflow: generate a counterfactual with **DiCE**, then hand it to `xains` to verbalize and score it. `xains` never searches for counterfactuals; it consumes pre-computed ones.

1. Train a model and generate a real counterfactual with DiCE (`total_CFs=1`).
2. Verbalize it two ways: **templated** (no LLM, reproducible) and **LLM** (richer prose).
3. Auto-extract the LLM narrative's claims and **grade** them on three CF-fidelity metrics.

For the feature-importance side of the library, see `feature_importance_narratives.ipynb`.

## Prerequisites

```bash
pip install xains dice-ml
```

`dice-ml` pulls in `scikit-learn`, `pandas`, and some gradient-boosting libraries (lightgbm, xgboost); it is a sizeable install. We also use `python-dotenv` to load the API key.

Create a `.env`:

```bash
ANTHROPIC_API_KEY="sk-ant-..."
```


In [1]:
import pandas as pd
from dotenv import load_dotenv
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

import dice_ml

from xains import (
    DatasetSchema,
    Explainer,
    ExplanationConfig,
    FeatureSchema,
    Modality,
    Prediction,
    TargetSchema,
    TabularContribution,
    TabularCounterfactual,
    TabularExplanationRequest,
    LLMNarrativeGenerator,
    TemplatedCounterfactualGenerator,
    grade_counterfactual,
    render_grades,
)
from xains.counterfactuals import build_scenarios
from xains.prompts import CounterfactualTabularPromptTemplate
from xains.config import DEFAULT_NARRATIVE_RULES
from xains.providers import AnthropicProvider

load_dotenv()

True

## Step 1 - Train a model on the raw German Credit data

We use a 30-row slice (`data/german_credit_sample.csv`) with its original (non-encoded) columns: 7 numeric and 13 categorical. A `Pipeline` one-hot-encodes inside the model, so DiCE and the schema can work with human-readable raw feature values (`checking_status = '<0'`) rather than one-hot dummies.


In [2]:
df = pd.read_csv("data/german_credit_sample.csv")
X_raw = df.drop(columns=["class"])
y = df["class"]

numeric_cols = X_raw.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = [c for c in X_raw.columns if c not in numeric_cols]

pipeline = Pipeline(steps=[
    ("pre", ColumnTransformer(transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols),
    ])),
    ("clf", RandomForestClassifier(n_estimators=100, random_state=42)),
])
pipeline.fit(X_raw, y)

print(f"Numeric features ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")
print(f"Accuracy on the slice: {pipeline.score(X_raw, y):.2f}")

Numeric features (7): ['duration', 'credit_amount', 'installment_commitment', 'residence_since', 'age', 'existing_credits', 'num_dependents']
Categorical features (13): ['checking_status', 'credit_history', 'purpose', 'savings_status', 'employment', 'personal_status', 'other_parties', 'property_magnitude', 'other_payment_plans', 'housing', 'job', 'own_telephone', 'foreign_worker']
Accuracy on the slice: 1.00


## Step 2 - Build the schema

One `FeatureSchema` per raw column: `numeric` for the 7 integer columns, `categorical` (with the observed category vocabulary) for the 13 others. The grader uses `dtype` to decide how to compare stated-vs-actual values: numeric uses `math.isclose`, categorical uses equality.


In [3]:
features_schema = []
for col in X_raw.columns:
    if col in numeric_cols:
        features_schema.append(FeatureSchema(
            name=col, dtype="numeric", description=f"Numeric feature: {col}",
        ))
    else:
        cats = sorted(str(v) for v in X_raw[col].unique())
        features_schema.append(FeatureSchema(
            name=col, dtype="categorical", description=f"Categorical feature: {col}",
            categories=cats,
        ))

schema = DatasetSchema(
    modality=Modality.TABULAR,
    name="german_credit",
    description="Predicts whether a loan applicant is a good or bad credit risk (OpenML credit-g).",
    target=TargetSchema(
        name="class",
        description="Credit risk classification.",
        classes={0: "Good credit risk", 1: "Bad credit risk"},
    ),
    features=features_schema,
)

## Step 3 - Generate a counterfactual with DiCE

We pick the first applicant the model predicts as **Bad credit risk** and ask DiCE for a single counterfactual that flips the prediction to **Good** (`total_CFs=1`, `desired_class=0`). DiCE returns one recipe -- which features to change, and to what. This is exactly what `xains` consumes: one counterfactual per request (ADR 0031), which may change several features.


In [4]:
data = dice_ml.Data(dataframe=df, continuous_features=numeric_cols, outcome_name="class")
dice_model = dice_ml.Model(model=pipeline, backend="sklearn")
dice = dice_ml.Dice(data, dice_model, method="random")

# First applicant the model predicts as Bad credit risk (class 1).
preds = pipeline.predict(X_raw)
bad_idx = next(i for i, p in enumerate(preds) if p == 1)
query = X_raw.iloc[[bad_idx]]
factual_proba = pipeline.predict_proba(query)[0]
print(f"Factual applicant (row {bad_idx}): Bad credit risk, P(Bad)={factual_proba[1]:.2f}")

cf_result = dice.generate_counterfactuals(query, total_CFs=1, desired_class=0)
cf_df = cf_result.cf_examples_list[0].final_cfs_df
cf_row = cf_df.iloc[0]

# Cast values to plain Python (DiCE returns numpy scalars / object dtype).
def _py(v):
    if isinstance(v, str):
        return v
    f = float(v)
    return int(f) if f.is_integer() else f

factual_features = {c: _py(query.iloc[0][c]) for c in X_raw.columns}
cf_features = {c: _py(cf_row[c]) for c in X_raw.columns}

print("\nDiCE counterfactual (changed features):")
for c in X_raw.columns:
    if factual_features[c] != cf_features[c]:
        print(f"  - {c}: {factual_features[c]} -> {cf_features[c]}")

Factual applicant (row 0): Bad credit risk, P(Bad)=0.77


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  4.49it/s]

100%|██████████| 1/1 [00:00<00:00,  4.46it/s]


DiCE counterfactual (changed features):
  - checking_status: <0 -> >=200
  - duration: 18 -> 45


## Step 4 - Build the xains request

We wrap DiCE's counterfactual in a `TabularCounterfactual` and assemble the `TabularExplanationRequest`. The request carries one placeholder `TabularContribution` only because the schema requires at least one; the counterfactual path never reads it.

`build_scenarios` derives the ground-truth changed-feature triples the metrics will score against.


In [5]:
cf = TabularCounterfactual(predicted_class=0, features=cf_features, method="DiCE")
request = TabularExplanationRequest(
    features=factual_features,
    prediction=Prediction(
        predicted_class=1,
        probabilities={0: float(factual_proba[0]), 1: float(factual_proba[1])},
    ),
    contributions=[
        # Placeholder only: the schema requires >=1 contribution; CF mode ignores it.
        TabularContribution(name=numeric_cols[0], value=factual_features[numeric_cols[0]], importance=0.0),
    ],
    counterfactual=cf,
    instance_id=f"row_{bad_idx}",
)

scenario = build_scenarios(request, schema)
print(f"Ground-truth changes ({len(scenario.changes)}):")
for chg in scenario.changes:
    print(f"  - {chg.name}: {chg.before} -> {chg.after}")

Ground-truth changes (2):
  - checking_status: <0 -> >=200
  - duration: 18 -> 45


## Step 5 - Path A: Templated narrative (no LLM, deterministic)

`TemplatedCounterfactualGenerator` turns the counterfactual into a one-sentence prose recipe with no LLM call. Deterministic given the same `(request, schema)` -- a reproducible baseline.


In [6]:
config = ExplanationConfig(
    mode="counterfactual",
    audience="end_user",
    tone="neutral",
    max_length_words=80,
    extract_narrative=False,
)

templated_result = TemplatedCounterfactualGenerator().generate(request, schema, config)
print(templated_result.text)

To change the prediction from Bad credit risk to Good credit risk, checking_status would need to change from <0 to >=200, and duration would need to change from 18 to 45.


## Step 6 - Path B: LLM narrative + automatic extraction

An `Explainer` with the CF prompt template and `mode="counterfactual"`. The same `llm` is the generator's LLM and the `judge_llm`; per ADR 0033 the Explainer dispatches extraction by mode, so `result.counterfactual_extraction` is populated automatically.


In [7]:
llm = AnthropicProvider(model="claude-haiku-4-5", max_tokens=512)

explainer = Explainer(
    schema=schema,
    generator=LLMNarrativeGenerator(
        prompt_template=CounterfactualTabularPromptTemplate(),
        llm=llm,
    ),
    config=ExplanationConfig(
        mode="counterfactual",
        audience="end_user",
        language="en",
        tone="neutral",
        max_length_words=80,
        extract_narrative=True,
        narrative_rules=DEFAULT_NARRATIVE_RULES,
    ),
    judge_llm=llm,
)

result = explainer.explain(request)

print("=== LLM narrative ===")
print(result.text)
print()
print("=== Auto-extracted CF claims ===")
ex = result.counterfactual_extraction
assert ex is not None, "explain() should populate counterfactual_extraction in CF mode"
for name, claim in ex.changes.items():
    print(f"  {name}: {claim.stated_before} -> {claim.stated_after} ({claim.stated_direction})")
if ex.invented:
    print(f"  invented mentions: {[c.narrative_name for c in ex.invented]}")

=== LLM narrative ===
# Counterfactual Explanation: Path to Better Credit Risk

Your application currently receives a "Bad credit risk" assessment. However, the model would shift this prediction to "Good credit risk" if two financial indicators improved. First, your checking account status would need to strengthen substantially, moving from its current position of **<0 to >=200**. This progression signals enhanced financial stability and available funds. Simultaneously, the loan duration would need to extend from **18 to 45** months, which distributes repayment obligations across a longer timeline. Together, these changes would demonstrate both improved liquidity and a more manageable payment structure, ultimately reversing the risk classification in your favor.

=== Auto-extracted CF claims ===
  checking_status: <0 -> >=200 (strengthen)
  duration: 18 -> 45 (extend)


## Step 7 - Grade the LLM narrative

`grade_counterfactual` scores the extraction against DiCE's ground-truth counterfactual:

* **`change_fidelity`** -- fraction of resolved claims about actually-changed features where BOTH `stated_before` AND `stated_after` match (strict). `None` when no resolved claim is about an actually-changed feature.
* **`coverage`** -- fraction of the CF's actually-changed features the narrative resolved.
* **`invented_features`** -- count of mentions the LLM could not resolve to a schema feature.

A faithful narrative scores near `1.00 / 1.00 / 0`. LLM output is non-deterministic, so exact numbers shift run-to-run.


In [8]:
cf_grades = grade_counterfactual(
    extraction=result.counterfactual_extraction,
    request=request,
    schema=schema,
)
print(render_grades(counterfactual=cf_grades))

Counterfactual fidelity
  change_fidelity ↑: 1.00
  coverage ↑: 1.00
  invented_features ↓: 0


## Summary

What we built end-to-end:

1. Trained a `Pipeline` (one-hot + RandomForest) on the raw German Credit slice.
2. Generated a real counterfactual with **DiCE** (`total_CFs=1`, `desired_class=0`).
3. Wrapped it in a `TabularCounterfactual` and built the request; `build_scenarios` surfaced the ground-truth changes.
4. Verbalized it two ways: `TemplatedCounterfactualGenerator` (deterministic, no LLM) and `LLMNarrativeGenerator` + `CounterfactualTabularPromptTemplate` (richer prose).
5. The LLM path auto-extracted the narrative's claims via `Explainer.explain()` (ADR 0033); we graded them with `grade_counterfactual`.

The pattern generalizes: bring a counterfactual from any source (DiCE, Wachter, Alibi, manual), and `xains` verbalizes and scores it. The library never searches for counterfactuals itself.

**Cost per run:** ~$0.001-0.003 (one Haiku narrative + one extraction call). The templated path is free.
